# Semantic Search Demo

This notebook provides an interactive demo of the SBERT-based semantic search for FMA tracks.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np

# Add project root to path
sys.path.append(os.path.abspath(".."))

from src.config import PROCESSED_DIR, FMA_METADATA_DIR
from src.embeddings.sbert import SentenceBERTEmbeddingGenerator
from src.indexing.faiss_index import FaissIndex
from src.metadata import load_tracks

### Load Index and Metadata

In [ ]:
sbert_gen = SentenceBERTEmbeddingGenerator()
index = FaissIndex.load(PROCESSED_DIR / "sbert_faiss.index")
tracks = load_tracks(FMA_METADATA_DIR)
print("System ready for search.")

### Search Function

In [ ]:
def search(query, k=5):
    q_emb = sbert_gen.embed_text([query])
    results = index.query(q_emb, k=k)
    
    df_res = []
    for tid, score in results:
        row = tracks.loc[tid]
        df_res.append({
            "track_id": tid,
            "title": row[('track', 'title')],
            "artist": row[('artist', 'name')],
            "genre": row[('track', 'genre_top')],
            "score": score
        })
    return pd.DataFrame(df_res)

# Example
search("songs about being alone in a crowd")

In [ ]:
search("upbeat dance music for a summer party")